# Assignment 3 — Object Detection

Bài toán phát hiện đối tượng (Object Detection) trên tập dữ liệu KITTI.

So sánh hai kiến trúc:
- **YOLOv8** (CNN-based detector)
- **DETR** (Transformer-based detector)

**Nhóm 4** — DL4CV

# 1. Setup & Imports

In [ ]:
!pip install ultralytics torchmetrics pycocotools torchvision transformers timm -q

In [ ]:
import os
import re
import json
import glob
import time
import random
import shutil
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.ops import box_convert

from torchmetrics.detection.mean_ap import MeanAveragePrecision

from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# KITTI Class Definitions
# ============================================================
# Chọn 8 lớp chính từ KITTI (bỏ DontCare, Misc)
KITTI_CLASSES = ['Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting',
                 'Cyclist', 'Tram', 'DontCare']

# Chỉ dùng các lớp có ý nghĩa cho detection
SELECTED_CLASSES = ['Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting',
                    'Cyclist', 'Tram']

CLASS2ID = {cls: i for i, cls in enumerate(SELECTED_CLASSES)}
ID2CLASS = {i: cls for cls, i in CLASS2ID.items()}
NUM_CLASSES = len(SELECTED_CLASSES)

print(f'Number of classes: {NUM_CLASSES}')
print(f'Class mapping: {CLASS2ID}')

In [ ]:
# ============================================================
# Dataset Paths — Cập nhật đường dẫn phù hợp với môi trường
# ============================================================
# KITTI Object Detection dataset structure:
#   data_object_image_2/training/image_2/  (*.png)
#   data_object_label_2/training/label_2/  (*.txt)

DATA_ROOT = '/kaggle/input/datasets/klemenko/kitti-dataset'
IMAGE_DIR = os.path.join(DATA_ROOT, 'data_object_image_2', 'training', 'image_2')
LABEL_DIR = os.path.join(DATA_ROOT, 'data_object_label_2', 'training', 'label_2')

# Kiểm tra thư mục tồn tại
assert os.path.isdir(IMAGE_DIR), f'Image directory not found: {IMAGE_DIR}'
assert os.path.isdir(LABEL_DIR), f'Label directory not found: {LABEL_DIR}'

image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, '*.png')))
label_files = sorted(glob.glob(os.path.join(LABEL_DIR, '*.txt')))

print(f'Found {len(image_files)} images')
print(f'Found {len(label_files)} label files')
print(f'Sample image: {image_files[0]}')
print(f'Sample label: {label_files[0]}')

# 2. Dataset Loading & Exploratory Data Analysis

In [ ]:
# ============================================================
# Parse KITTI Label Format
# ============================================================
# Mỗi dòng trong file label KITTI có format:
# type truncated occluded alpha bbox(x1,y1,x2,y2) dimensions(h,w,l) location(x,y,z) rotation_y

def parse_kitti_label(label_path):
    """Parse một file label KITTI, trả về list các annotation dict."""
    annotations = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue
            cls_name = parts[0]
            if cls_name not in SELECTED_CLASSES:
                continue
            annotations.append({
                'class': cls_name,
                'class_id': CLASS2ID[cls_name],
                'truncated': float(parts[1]),
                'occluded': int(parts[2]),
                'bbox': [float(parts[4]), float(parts[5]),
                         float(parts[6]), float(parts[7])],  # x1, y1, x2, y2
            })
    return annotations

# Parse tất cả labels
all_annotations = {}
for lf in label_files:
    img_id = os.path.splitext(os.path.basename(lf))[0]
    all_annotations[img_id] = parse_kitti_label(lf)

# Thống kê
total_objects = sum(len(v) for v in all_annotations.values())
images_with_objects = sum(1 for v in all_annotations.values() if len(v) > 0)

print(f'Total annotated objects: {total_objects}')
print(f'Images with objects: {images_with_objects} / {len(all_annotations)}')
print(f'Average objects per image: {total_objects / len(all_annotations):.1f}')

In [ ]:
# ============================================================
# EDA: Class Distribution
# ============================================================
class_counts = Counter()
for anns in all_annotations.values():
    for ann in anns:
        class_counts[ann['class']] += 1

classes_sorted = sorted(class_counts.keys(), key=lambda x: class_counts[x], reverse=True)
counts_sorted = [class_counts[c] for c in classes_sorted]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('KITTI Dataset — Class Distribution', fontsize=15, fontweight='bold')

# Bar chart
colors = sns.color_palette('husl', len(classes_sorted))
axes[0].barh(classes_sorted, counts_sorted, color=colors, edgecolor='white')
axes[0].set_xlabel('Number of Instances')
axes[0].set_title('Object Count per Class')
for i, v in enumerate(counts_sorted):
    axes[0].text(v + 50, i, str(v), va='center', fontweight='bold')

# Pie chart
axes[1].pie(counts_sorted, labels=classes_sorted, autopct='%1.1f%%',
            startangle=140, colors=colors)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()

print('\nClass distribution:')
for cls, cnt in zip(classes_sorted, counts_sorted):
    print(f'  {cls:<16}: {cnt:>5} ({cnt/total_objects*100:.1f}%)')

In [ ]:
# ============================================================
# EDA: Image Size Statistics
# ============================================================
widths, heights = [], []
for img_path in image_files[:500]:  # Sample 500 ảnh cho nhanh
    img = Image.open(img_path)
    widths.append(img.width)
    heights.append(img.height)

print(f'Image size stats (sampled {len(widths)} images):')
print(f'  Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'  Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

In [ ]:
# ============================================================
# EDA: Bounding Box Size Distribution
# ============================================================
bbox_widths, bbox_heights, bbox_areas, bbox_classes = [], [], [], []
for anns in all_annotations.values():
    for ann in anns:
        x1, y1, x2, y2 = ann['bbox']
        w = x2 - x1
        h = y2 - y1
        bbox_widths.append(w)
        bbox_heights.append(h)
        bbox_areas.append(w * h)
        bbox_classes.append(ann['class'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Bounding Box Size Distribution', fontsize=15, fontweight='bold')

axes[0].hist(bbox_widths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('BBox Width')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(np.mean(bbox_widths), color='red', linestyle='--',
                label=f'Mean={np.mean(bbox_widths):.0f}')
axes[0].legend()

axes[1].hist(bbox_heights, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('BBox Height')
axes[1].set_xlabel('Height (pixels)')
axes[1].axvline(np.mean(bbox_heights), color='red', linestyle='--',
                label=f'Mean={np.mean(bbox_heights):.0f}')
axes[1].legend()

axes[2].hist(bbox_areas, bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
axes[2].set_title('BBox Area')
axes[2].set_xlabel('Area (pixels²)')
axes[2].axvline(np.mean(bbox_areas), color='red', linestyle='--',
                label=f'Mean={np.mean(bbox_areas):.0f}')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EDA: Sample Images with Bounding Boxes
# ============================================================
CLASS_COLORS = {cls: plt.cm.tab10(i / NUM_CLASSES) for i, cls in enumerate(SELECTED_CLASSES)}

def visualize_sample(img_path, annotations, ax):
    """Vẽ ảnh kèm bounding boxes."""
    img = Image.open(img_path)
    ax.imshow(img)
    for ann in annotations:
        x1, y1, x2, y2 = ann['bbox']
        color = CLASS_COLORS[ann['class']]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, ann['class'], color=color,
                fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    ax.axis('off')

# Chọn 6 ảnh có nhiều object
sorted_ids = sorted(all_annotations.keys(),
                     key=lambda x: len(all_annotations[x]), reverse=True)
sample_ids = sorted_ids[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Sample KITTI Images with Ground Truth Bounding Boxes',
             fontsize=15, fontweight='bold')

for idx, img_id in enumerate(sample_ids):
    ax = axes[idx // 3, idx % 3]
    img_path = os.path.join(IMAGE_DIR, f'{img_id}.png')
    visualize_sample(img_path, all_annotations[img_id], ax)
    ax.set_title(f'Image {img_id} ({len(all_annotations[img_id])} objects)', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EDA: Objects per Image Distribution
# ============================================================
objects_per_image = [len(v) for v in all_annotations.values()]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(objects_per_image, bins=range(0, max(objects_per_image) + 2),
        color='teal', edgecolor='white', alpha=0.8)
ax.set_title('Number of Objects per Image', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Objects')
ax.set_ylabel('Frequency')
ax.axvline(np.mean(objects_per_image), color='red', linestyle='--',
           label=f'Mean={np.mean(objects_per_image):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

# 3. Data Preprocessing

In [ ]:
# ============================================================
# Train / Val Split (80/20)
# ============================================================
# Chỉ lấy các ảnh có ít nhất 1 object
valid_ids = [img_id for img_id, anns in all_annotations.items() if len(anns) > 0]
print(f'Images with objects: {len(valid_ids)}')

train_ids, val_ids = train_test_split(valid_ids, test_size=0.2, random_state=SEED)
print(f'Train: {len(train_ids)} images')
print(f'Val:   {len(val_ids)} images')

In [ ]:
# ============================================================
# Convert KITTI -> YOLO Format (cho YOLOv8)
# ============================================================
# YOLO format: class_id x_center y_center width height (normalized 0-1)
# Ghi vào /kaggle/working vì /kaggle/input là read-only

YOLO_DIR = '/kaggle/working/kitti_yolo_format'
os.makedirs(os.path.join(YOLO_DIR, 'images', 'train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DIR, 'images', 'val'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DIR, 'labels', 'train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DIR, 'labels', 'val'), exist_ok=True)

def convert_to_yolo(img_id, split):
    """Chuyển đổi annotation từ KITTI sang YOLO format."""
    img_path = os.path.join(IMAGE_DIR, f'{img_id}.png')
    img = Image.open(img_path)
    img_w, img_h = img.size

    # Symlink ảnh thay vì copy để tiết kiệm dung lượng và thời gian
    dst_img = os.path.join(YOLO_DIR, 'images', split, f'{img_id}.png')
    if not os.path.exists(dst_img):
        os.symlink(img_path, dst_img)

    # Tạo file label YOLO
    label_path = os.path.join(YOLO_DIR, 'labels', split, f'{img_id}.txt')
    with open(label_path, 'w') as f:
        for ann in all_annotations[img_id]:
            x1, y1, x2, y2 = ann['bbox']
            # Convert to YOLO: x_center, y_center, w, h (normalized)
            x_center = ((x1 + x2) / 2) / img_w
            y_center = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            f.write(f'{ann["class_id"]} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n')

print('Converting to YOLO format...')
for img_id in train_ids:
    convert_to_yolo(img_id, 'train')
for img_id in val_ids:
    convert_to_yolo(img_id, 'val')

print(f'YOLO format created at: {YOLO_DIR}')
print(f'  Train images: {len(os.listdir(os.path.join(YOLO_DIR, "images", "train")))}')
print(f'  Val images:   {len(os.listdir(os.path.join(YOLO_DIR, "images", "val")))}')

In [ ]:
# ============================================================
# Tạo YOLO dataset.yaml
# ============================================================
yaml_content = f"""path: {os.path.abspath(YOLO_DIR)}
train: images/train
val: images/val

nc: {NUM_CLASSES}
names: {SELECTED_CLASSES}
"""

yaml_path = os.path.join(YOLO_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f'Created dataset.yaml at: {yaml_path}')
print(yaml_content)

In [ ]:
# ============================================================
# Convert KITTI -> COCO JSON Format (cho DETR)
# ============================================================

def create_coco_json(img_ids, split_name):
    """Tạo file COCO JSON annotation từ KITTI labels."""
    coco = {
        'images': [],
        'annotations': [],
        'categories': [{'id': i, 'name': cls} for cls, i in CLASS2ID.items()]
    }
    ann_id = 0

    for img_id in img_ids:
        img_path = os.path.join(IMAGE_DIR, f'{img_id}.png')
        img = Image.open(img_path)
        img_w, img_h = img.size

        img_info = {
            'id': int(img_id),
            'file_name': f'{img_id}.png',
            'width': img_w,
            'height': img_h
        }
        coco['images'].append(img_info)

        for ann in all_annotations[img_id]:
            x1, y1, x2, y2 = ann['bbox']
            w = x2 - x1
            h = y2 - y1
            coco['annotations'].append({
                'id': ann_id,
                'image_id': int(img_id),
                'category_id': ann['class_id'],
                'bbox': [x1, y1, w, h],  # COCO format: x, y, w, h
                'area': w * h,
                'iscrowd': 0
            })
            ann_id += 1

    # Ghi vào /kaggle/working vì /kaggle/input là read-only
    out_path = os.path.join('/kaggle/working', f'{split_name}_coco.json')
    with open(out_path, 'w') as f:
        json.dump(coco, f)

    print(f'Created {out_path}: {len(coco["images"])} images, {len(coco["annotations"])} annotations')
    return out_path

train_coco_path = create_coco_json(train_ids, 'train')
val_coco_path = create_coco_json(val_ids, 'val')

In [ ]:
# ============================================================
# Verification: Kiểm tra round-trip conversion
# ============================================================
sample_id = train_ids[0]
print(f'Verifying conversion for image {sample_id}:')

# KITTI original
kitti_anns = all_annotations[sample_id]
print(f'  KITTI annotations: {len(kitti_anns)} objects')
for ann in kitti_anns[:3]:
    print(f'    {ann["class"]}: bbox={ann["bbox"]}')

# YOLO format
yolo_label = os.path.join(YOLO_DIR, 'labels', 'train', f'{sample_id}.txt')
with open(yolo_label, 'r') as f:
    yolo_lines = f.readlines()
print(f'  YOLO labels: {len(yolo_lines)} lines')
for line in yolo_lines[:3]:
    print(f'    {line.strip()}')

# COCO format
with open(train_coco_path, 'r') as f:
    coco_data = json.load(f)
coco_anns = [a for a in coco_data['annotations'] if a['image_id'] == int(sample_id)]
print(f'  COCO annotations: {len(coco_anns)} objects')
for ann in coco_anns[:3]:
    cat_name = [c['name'] for c in coco_data['categories'] if c['id'] == ann['category_id']][0]
    print(f'    {cat_name}: bbox={ann["bbox"]}')

print('\nConversion verified!')

# 4. YOLOv8 — Training & Evaluation

In [ ]:
# ============================================================
# 4.1. YOLOv8 Training
# ============================================================
from ultralytics import YOLO

YOLO_PROJECT = '/kaggle/working/runs/yolo'
YOLO_RUN_NAME = 'kitti_yolov8s'
YOLO_EPOCHS = 50   # YOLOv8 hội tụ nhanh, 50 epochs là đủ
DETR_EPOCHS = 20   # DETR chậm hơn nhiều, 20 epochs để tiết kiệm thời gian trên Kaggle

# Cache YOLOv8s weights để không phải download lại
YOLO_CACHE = '/kaggle/working/models/yolov8s.pt'
os.makedirs(os.path.dirname(YOLO_CACHE), exist_ok=True)

if os.path.exists(YOLO_CACHE):
    print(f'Loading cached YOLOv8s from {YOLO_CACHE}')
    yolo_model = YOLO(YOLO_CACHE)
else:
    print('Downloading YOLOv8s (first time only)...')
    yolo_model = YOLO('yolov8s.pt')
    shutil.copy2('yolov8s.pt', YOLO_CACHE)
    print(f'Saved to {YOLO_CACHE}')

# Fine-tune trên KITTI
yolo_results = yolo_model.train(
    data=yaml_path,
    epochs=YOLO_EPOCHS,
    imgsz=640,
    batch=16,
    lr0=0.01,
    lrf=0.01,
    optimizer='SGD',
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    seed=SEED,
    project=YOLO_PROJECT,
    name=YOLO_RUN_NAME,
    exist_ok=True,
    verbose=True,
)

In [ ]:
# ============================================================
# 4.2. YOLOv8 — Evaluate with torchmetrics mAP
# ============================================================

# Load best YOLOv8 weights
best_yolo_path = os.path.join(YOLO_PROJECT, YOLO_RUN_NAME, 'weights', 'best.pt')
best_yolo = YOLO(best_yolo_path)

# Chạy prediction trên tập val
yolo_preds_list = []
yolo_targets_list = []

for img_id in val_ids:
    img_path = os.path.join(IMAGE_DIR, f'{img_id}.png')
    img = Image.open(img_path)
    img_w, img_h = img.size

    # Prediction
    results = best_yolo(img_path, verbose=False)
    result = results[0]

    boxes = result.boxes
    if len(boxes) > 0:
        pred_dict = {
            'boxes': boxes.xyxy.cpu(),
            'scores': boxes.conf.cpu(),
            'labels': boxes.cls.cpu().int(),
        }
    else:
        pred_dict = {
            'boxes': torch.zeros((0, 4)),
            'scores': torch.zeros(0),
            'labels': torch.zeros(0, dtype=torch.int),
        }

    # Ground truth
    gt_anns = all_annotations[img_id]
    if len(gt_anns) > 0:
        gt_boxes = torch.tensor([a['bbox'] for a in gt_anns], dtype=torch.float32)
        gt_labels = torch.tensor([a['class_id'] for a in gt_anns], dtype=torch.int)
    else:
        gt_boxes = torch.zeros((0, 4))
        gt_labels = torch.zeros(0, dtype=torch.int)

    target_dict = {
        'boxes': gt_boxes,
        'labels': gt_labels,
    }

    yolo_preds_list.append(pred_dict)
    yolo_targets_list.append(target_dict)

# Compute mAP
yolo_metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
yolo_metric.update(yolo_preds_list, yolo_targets_list)
yolo_map_results = yolo_metric.compute()

print('=' * 60)
print('           YOLOv8 — Evaluation Results')
print('=' * 60)
print(f'  mAP@0.50      : {yolo_map_results["map_50"]:.4f}')
print(f'  mAP@0.75      : {yolo_map_results["map_75"]:.4f}')
print(f'  mAP@0.50:0.95 : {yolo_map_results["map"]:.4f}')

# Per-class AP
if 'map_per_class' in yolo_map_results and yolo_map_results['map_per_class'].numel() > 0:
    print('\n  Per-class AP@0.50:')
    for i, cls_name in enumerate(SELECTED_CLASSES):
        if i < len(yolo_map_results['map_per_class']):
            print(f'    {cls_name:<16}: {yolo_map_results["map_per_class"][i]:.4f}')

In [ ]:
# ============================================================
# 4.3. YOLOv8 Training Curves
# ============================================================
yolo_csv = os.path.join(YOLO_PROJECT, YOLO_RUN_NAME, 'results.csv')
if os.path.exists(yolo_csv):
    yolo_df = pd.read_csv(yolo_csv)
    yolo_df.columns = yolo_df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('YOLOv8 Training Curves', fontsize=15, fontweight='bold')

    # Box Loss
    axes[0].plot(yolo_df['train/box_loss'], label='Train', color='#e74c3c', linewidth=2)
    axes[0].plot(yolo_df['val/box_loss'], label='Val', color='#3498db', linewidth=2, linestyle='--')
    axes[0].set_title('Box Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Class Loss
    axes[1].plot(yolo_df['train/cls_loss'], label='Train', color='#e74c3c', linewidth=2)
    axes[1].plot(yolo_df['val/cls_loss'], label='Val', color='#3498db', linewidth=2, linestyle='--')
    axes[1].set_title('Classification Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # mAP
    if 'metrics/mAP50(B)' in yolo_df.columns:
        axes[2].plot(yolo_df['metrics/mAP50(B)'], label='mAP@0.50', color='#2ecc71', linewidth=2)
    if 'metrics/mAP50-95(B)' in yolo_df.columns:
        axes[2].plot(yolo_df['metrics/mAP50-95(B)'], label='mAP@0.50:0.95', color='#e67e22', linewidth=2, linestyle='--')
    axes[2].set_title('mAP Metrics')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('mAP')
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print(f'Training results CSV not found at {yolo_csv}')

In [ ]:
# ============================================================
# 4.4. YOLOv8 — FPS Measurement
# ============================================================
def measure_fps_yolo(model, image_paths, num_warmup=10, num_runs=100):
    """Đo FPS của YOLOv8."""
    # Warmup
    for i in range(min(num_warmup, len(image_paths))):
        _ = model(image_paths[i], verbose=False)

    if device.type == 'cuda':
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()

        for i in range(num_runs):
            idx = i % len(image_paths)
            _ = model(image_paths[idx], verbose=False)

        end.record()
        torch.cuda.synchronize()
        elapsed_ms = start.elapsed_time(end)
    else:
        t0 = time.time()
        for i in range(num_runs):
            idx = i % len(image_paths)
            _ = model(image_paths[idx], verbose=False)
        elapsed_ms = (time.time() - t0) * 1000

    fps = num_runs / (elapsed_ms / 1000)
    return fps, elapsed_ms / num_runs

val_image_paths = [os.path.join(IMAGE_DIR, f'{img_id}.png') for img_id in val_ids[:100]]
yolo_fps, yolo_latency = measure_fps_yolo(best_yolo, val_image_paths)

print(f'YOLOv8 Inference Speed:')
print(f'  FPS: {yolo_fps:.1f}')
print(f'  Latency: {yolo_latency:.1f} ms/image')

In [ ]:
# ============================================================
# 4.5. YOLOv8 — Visual Predictions
# ============================================================
sample_val_ids = val_ids[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLOv8 Predictions on Validation Set', fontsize=15, fontweight='bold')

for idx, img_id in enumerate(sample_val_ids):
    ax = axes[idx // 3, idx % 3]
    img_path = os.path.join(IMAGE_DIR, f'{img_id}.png')
    img = Image.open(img_path)
    ax.imshow(img)

    results = best_yolo(img_path, verbose=False)
    result = results[0]

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls_id = int(box.cls[0].cpu())
        conf = float(box.conf[0].cpu())
        cls_name = SELECTED_CLASSES[cls_id] if cls_id < len(SELECTED_CLASSES) else f'cls_{cls_id}'
        color = CLASS_COLORS.get(cls_name, 'yellow')

        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f'{cls_name} {conf:.2f}', color=color,
                fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

    ax.set_title(f'Image {img_id}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

# 5. DETR — Training & Evaluation

In [ ]:
# ============================================================
# 5.0. GPU Cleanup
# ============================================================
# Giải phóng bộ nhớ GPU trước khi train DETR
if device.type == 'cuda':
    if 'best_yolo' in dir():
        del best_yolo
    torch.cuda.empty_cache()
    print(f'GPU memory freed. Available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

In [ ]:
# ============================================================
# 5.1. DETR Dataset Class
# ============================================================
from transformers import DetrImageProcessor

DETR_MODEL_NAME = 'facebook/detr-resnet-50'
DETR_CACHE = '/kaggle/working/models/detr-resnet-50'

# Cache DETR processor để không phải download lại
if os.path.exists(os.path.join(DETR_CACHE, 'preprocessor_config.json')):
    print(f'Loading cached DETR processor from {DETR_CACHE}')
    detr_processor = DetrImageProcessor.from_pretrained(DETR_CACHE)
else:
    print('Downloading DETR processor (first time only)...')
    detr_processor = DetrImageProcessor.from_pretrained(DETR_MODEL_NAME)
    os.makedirs(DETR_CACHE, exist_ok=True)
    detr_processor.save_pretrained(DETR_CACHE)
    print(f'Saved to {DETR_CACHE}')

class KITTIDetrDataset(Dataset):
    """Dataset class cho DETR, trả về format phù hợp với HuggingFace DETR."""

    def __init__(self, img_ids, annotations, image_dir, processor):
        self.img_ids = img_ids
        self.annotations = annotations
        self.image_dir = image_dir
        self.processor = processor

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = os.path.join(self.image_dir, f'{img_id}.png')
        image = Image.open(img_path).convert('RGB')
        img_w, img_h = image.size

        anns = self.annotations[img_id]

        # DETR expects COCO format annotations
        target = {
            'image_id': int(img_id),
            'annotations': []
        }

        for ann in anns:
            x1, y1, x2, y2 = ann['bbox']
            # COCO format: [x, y, width, height]
            target['annotations'].append({
                'bbox': [x1, y1, x2 - x1, y2 - y1],
                'category_id': ann['class_id'],
                'area': (x2 - x1) * (y2 - y1),
                'iscrowd': 0,
            })

        # Process with DETR processor
        encoding = self.processor(
            images=image,
            annotations=[target],
            return_tensors='pt'
        )

        # Remove batch dimension
        pixel_values = encoding['pixel_values'].squeeze(0)
        labels = encoding['labels'][0]

        return pixel_values, labels

print(f'DETR processor ready')

In [ ]:
# ============================================================
# 5.2. DETR Collate Function & DataLoaders
# ============================================================

def detr_collate_fn(batch):
    """Custom collate function cho DETR — pad variable-size images và tạo pixel_mask."""
    pixel_values_list = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    # Pad tất cả ảnh về cùng kích thước (max trong batch)
    max_h = max(pv.shape[1] for pv in pixel_values_list)
    max_w = max(pv.shape[2] for pv in pixel_values_list)

    padded = []
    masks = []
    for pv in pixel_values_list:
        pad_h = max_h - pv.shape[1]
        pad_w = max_w - pv.shape[2]
        padded_pv = torch.nn.functional.pad(pv, (0, pad_w, 0, pad_h), value=0)
        padded.append(padded_pv)
        # pixel_mask: True = valid pixel, False = padding
        mask = torch.zeros((max_h, max_w), dtype=torch.bool)
        mask[:pv.shape[1], :pv.shape[2]] = True
        masks.append(mask)

    return torch.stack(padded), torch.stack(masks), labels

# Create datasets
train_detr_dataset = KITTIDetrDataset(train_ids, all_annotations, IMAGE_DIR, detr_processor)
val_detr_dataset = KITTIDetrDataset(val_ids, all_annotations, IMAGE_DIR, detr_processor)

DETR_BATCH_SIZE = 4  # DETR cần nhiều GPU memory hơn

train_detr_loader = DataLoader(
    train_detr_dataset, batch_size=DETR_BATCH_SIZE, shuffle=True,
    collate_fn=detr_collate_fn, num_workers=2, pin_memory=True
)
val_detr_loader = DataLoader(
    val_detr_dataset, batch_size=DETR_BATCH_SIZE, shuffle=False,
    collate_fn=detr_collate_fn, num_workers=2, pin_memory=True
)

# Verify
pixel_values, pixel_mask, labels = next(iter(train_detr_loader))
print(f'Batch pixel_values shape: {pixel_values.shape}')
print(f'Batch pixel_mask shape: {pixel_mask.shape}')
print(f'Number of label dicts: {len(labels)}')
print(f'Sample label keys: {labels[0].keys()}')

In [ ]:
# ============================================================
# 5.3. DETR Model Setup
# ============================================================
from transformers import DetrForObjectDetection

# Cache DETR model để không phải download lại
if os.path.exists(os.path.join(DETR_CACHE, 'model.safetensors')):
    print(f'Loading cached DETR model from {DETR_CACHE}')
    detr_model = DetrForObjectDetection.from_pretrained(
        DETR_CACHE,
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True
    )
else:
    print('Downloading DETR model (first time only)...')
    detr_model = DetrForObjectDetection.from_pretrained(
        DETR_MODEL_NAME,
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True
    )
    detr_model.save_pretrained(DETR_CACHE)
    print(f'Saved to {DETR_CACHE}')

detr_model = detr_model.to(device)

total_params = sum(p.numel() for p in detr_model.parameters())
trainable_params = sum(p.numel() for p in detr_model.parameters() if p.requires_grad)

print(f'DETR Model ready')
print(f'  Total parameters    : {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}')
print(f'  Num classes: {NUM_CLASSES}')

In [ ]:
# ============================================================
# 5.4. DETR Training Loop
# ============================================================
DETR_LR = 1e-4
DETR_LR_BACKBONE = 1e-5
DETR_WEIGHT_DECAY = 1e-4

# Separate learning rates for backbone vs rest
param_dicts = [
    {'params': [p for n, p in detr_model.named_parameters()
                if 'backbone' not in n and p.requires_grad],
     'lr': DETR_LR},
    {'params': [p for n, p in detr_model.named_parameters()
                if 'backbone' in n and p.requires_grad],
     'lr': DETR_LR_BACKBONE},
]

optimizer = optim.AdamW(param_dicts, weight_decay=DETR_WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

print(f'DETR Training Config:')
print(f'  Epochs: {DETR_EPOCHS}')
print(f'  LR (head): {DETR_LR}, LR (backbone): {DETR_LR_BACKBONE}')
print(f'  Batch size: {DETR_BATCH_SIZE}')
print(f'  Weight decay: {DETR_WEIGHT_DECAY}')

detr_history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
detr_save_path = '/kaggle/working/best_detr_model.pt'

print('\n' + '=' * 65)
print('           DETR FINE-TUNING — TRAINING START')
print('=' * 65)

for epoch in range(1, DETR_EPOCHS + 1):
    t0 = time.time()

    # ── Training ──
    detr_model.train()
    epoch_train_loss = 0

    for step, (pixel_values, pixel_mask, labels) in enumerate(train_detr_loader):
        pixel_values = pixel_values.to(device)
        pixel_mask = pixel_mask.to(device)
        labels = [{k: v.to(device) for k, v in t.items()} for t in labels]

        outputs = detr_model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(detr_model.parameters(), max_norm=0.1)
        optimizer.step()

        epoch_train_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch} Step [{step+1}/{len(train_detr_loader)}] | loss: {epoch_train_loss/(step+1):.4f}')

    avg_train_loss = epoch_train_loss / len(train_detr_loader)

    # ── Validation ──
    detr_model.eval()
    epoch_val_loss = 0

    with torch.no_grad():
        for pixel_values, pixel_mask, labels in val_detr_loader:
            pixel_values = pixel_values.to(device)
            pixel_mask = pixel_mask.to(device)
            labels = [{k: v.to(device) for k, v in t.items()} for t in labels]

            outputs = detr_model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
            epoch_val_loss += outputs.loss.item()

    avg_val_loss = epoch_val_loss / len(val_detr_loader)
    scheduler.step()

    detr_history['train_loss'].append(avg_train_loss)
    detr_history['val_loss'].append(avg_val_loss)

    elapsed = time.time() - t0
    print(f'\n  Epoch [{epoch}/{DETR_EPOCHS}] | Time: {elapsed/60:.1f} min')
    print(f'    Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(detr_model.state_dict(), detr_save_path)
        print(f'    Best model saved (val loss: {best_val_loss:.4f})')

print('\n' + '=' * 65)
print(f'  Training complete! Best Val Loss: {best_val_loss:.4f}')
print('=' * 65)

In [ ]:
# ============================================================
# 5.5. DETR Training Curves
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
epochs_range = range(1, len(detr_history['train_loss']) + 1)

ax.plot(epochs_range, detr_history['train_loss'], 'o-', label='Train Loss',
        color='#e74c3c', linewidth=2)
ax.plot(epochs_range, detr_history['val_loss'], 's--', label='Val Loss',
        color='#3498db', linewidth=2)
ax.set_title('DETR Training Curves', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.6. DETR — Evaluate with torchmetrics mAP
# ============================================================

# Load best DETR model
detr_model.load_state_dict(torch.load(detr_save_path, map_location=device, weights_only=True))
detr_model.eval()

detr_preds_list = []
detr_targets_list = []

with torch.no_grad():
    for pixel_values, pixel_mask, labels in val_detr_loader:
        pixel_values = pixel_values.to(device)
        pixel_mask = pixel_mask.to(device)

        outputs = detr_model(pixel_values=pixel_values, pixel_mask=pixel_mask)

        # Post-process DETR outputs — dùng kích thước thực từ label['size']
        target_sizes = torch.stack([label['size'] for label in labels]).to(device)

        results = detr_processor.post_process_object_detection(
            outputs, threshold=0.5, target_sizes=target_sizes
        )

        for i, (result, label) in enumerate(zip(results, labels)):
            # Predictions
            pred_dict = {
                'boxes': result['boxes'].cpu(),
                'scores': result['scores'].cpu(),
                'labels': result['labels'].cpu().int(),
            }
            detr_preds_list.append(pred_dict)

            # Ground truth — convert from normalized cxcywh to xyxy
            gt_boxes_cxcywh = label['boxes']
            if gt_boxes_cxcywh.numel() > 0:
                img_h, img_w = label['size'].tolist()
                gt_boxes_cxcywh = gt_boxes_cxcywh.clone()
                gt_boxes_cxcywh[:, 0] *= img_w
                gt_boxes_cxcywh[:, 1] *= img_h
                gt_boxes_cxcywh[:, 2] *= img_w
                gt_boxes_cxcywh[:, 3] *= img_h
                gt_boxes_xyxy = box_convert(gt_boxes_cxcywh, 'cxcywh', 'xyxy')
            else:
                gt_boxes_xyxy = torch.zeros((0, 4))

            target_dict = {
                'boxes': gt_boxes_xyxy.cpu(),
                'labels': label['class_labels'].cpu().int(),
            }
            detr_targets_list.append(target_dict)

# Compute mAP
detr_metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
detr_metric.update(detr_preds_list, detr_targets_list)
detr_map_results = detr_metric.compute()

print('=' * 60)
print('           DETR — Evaluation Results')
print('=' * 60)
print(f'  mAP@0.50      : {detr_map_results["map_50"]:.4f}')
print(f'  mAP@0.75      : {detr_map_results["map_75"]:.4f}')
print(f'  mAP@0.50:0.95 : {detr_map_results["map"]:.4f}')

# Per-class AP
if 'map_per_class' in detr_map_results and detr_map_results['map_per_class'].numel() > 0:
    print('\n  Per-class AP@0.50:')
    for i, cls_name in enumerate(SELECTED_CLASSES):
        if i < len(detr_map_results['map_per_class']):
            print(f'    {cls_name:<16}: {detr_map_results["map_per_class"][i]:.4f}')

In [ ]:
# ============================================================
# 5.7. DETR — FPS Measurement
# ============================================================
def measure_fps_detr(model, processor, image_dir, img_ids, device,
                     num_warmup=10, num_runs=100):
    """Đo FPS của DETR."""
    model.eval()

    # Warmup
    for i in range(min(num_warmup, len(img_ids))):
        img_path = os.path.join(image_dir, f'{img_ids[i]}.png')
        image = Image.open(img_path).convert('RGB')
        inputs = processor(images=image, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            _ = model(**inputs)

    if device.type == 'cuda':
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()

        for i in range(num_runs):
            idx = i % len(img_ids)
            img_path = os.path.join(image_dir, f'{img_ids[idx]}.png')
            image = Image.open(img_path).convert('RGB')
            inputs = processor(images=image, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                _ = model(**inputs)

        end.record()
        torch.cuda.synchronize()
        elapsed_ms = start.elapsed_time(end)
    else:
        t0 = time.time()
        for i in range(num_runs):
            idx = i % len(img_ids)
            img_path = os.path.join(image_dir, f'{img_ids[idx]}.png')
            image = Image.open(img_path).convert('RGB')
            inputs = processor(images=image, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                _ = model(**inputs)
        elapsed_ms = (time.time() - t0) * 1000

    fps = num_runs / (elapsed_ms / 1000)
    return fps, elapsed_ms / num_runs

detr_fps, detr_latency = measure_fps_detr(
    detr_model, detr_processor, IMAGE_DIR, val_ids[:100], device
)

print(f'DETR Inference Speed:')
print(f'  FPS: {detr_fps:.1f}')
print(f'  Latency: {detr_latency:.1f} ms/image')

In [ ]:
# ============================================================
# 5.8. DETR — Visual Predictions
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('DETR Predictions on Validation Set', fontsize=15, fontweight='bold')

for idx, img_id in enumerate(sample_val_ids):
    ax = axes[idx // 3, idx % 3]
    img_path = os.path.join(IMAGE_DIR, f'{img_id}.png')
    image = Image.open(img_path).convert('RGB')
    ax.imshow(image)

    # Run DETR
    inputs = detr_processor(images=image, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = detr_model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]], device=device)
    results = detr_processor.post_process_object_detection(
        outputs, threshold=0.5, target_sizes=target_sizes
    )[0]

    for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
        x1, y1, x2, y2 = box.cpu().numpy()
        cls_id = label.item()
        conf = score.item()
        cls_name = SELECTED_CLASSES[cls_id] if cls_id < len(SELECTED_CLASSES) else f'cls_{cls_id}'
        color = CLASS_COLORS.get(cls_name, 'yellow')

        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f'{cls_name} {conf:.2f}', color=color,
                fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

    ax.set_title(f'Image {img_id}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

# 6. Comparison & Visualization

In [ ]:
# ============================================================
# 6.1. Metrics Comparison Table
# ============================================================
comparison_data = {
    'Metric': ['mAP@0.50', 'mAP@0.75', 'mAP@0.50:0.95', 'FPS', 'Latency (ms)'],
    'YOLOv8s': [
        f'{yolo_map_results["map_50"]:.4f}',
        f'{yolo_map_results["map_75"]:.4f}',
        f'{yolo_map_results["map"]:.4f}',
        f'{yolo_fps:.1f}',
        f'{yolo_latency:.1f}',
    ],
    'DETR': [
        f'{detr_map_results["map_50"]:.4f}',
        f'{detr_map_results["map_75"]:.4f}',
        f'{detr_map_results["map"]:.4f}',
        f'{detr_fps:.1f}',
        f'{detr_latency:.1f}',
    ],
}

comparison_df = pd.DataFrame(comparison_data)

print('\n' + '=' * 60)
print('        YOLOv8 vs DETR — Performance Comparison')
print('=' * 60)
print(comparison_df.to_string(index=False))
print('=' * 60)

In [ ]:
# ============================================================
# 6.2. Per-class AP Comparison Chart
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

x = np.arange(NUM_CLASSES)
width = 0.35

yolo_per_class = yolo_map_results.get('map_per_class', torch.zeros(NUM_CLASSES))
detr_per_class = detr_map_results.get('map_per_class', torch.zeros(NUM_CLASSES))

# Đảm bảo đủ length
yolo_ap = [yolo_per_class[i].item() if i < len(yolo_per_class) else 0 for i in range(NUM_CLASSES)]
detr_ap = [detr_per_class[i].item() if i < len(detr_per_class) else 0 for i in range(NUM_CLASSES)]

bars1 = ax.bar(x - width/2, yolo_ap, width, label='YOLOv8s', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, detr_ap, width, label='DETR', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('AP@0.50:0.95', fontsize=12)
ax.set_title('Per-class AP Comparison: YOLOv8 vs DETR', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(SELECTED_CLASSES, rotation=30, ha='right')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# Giá trị trên mỗi bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.3. Side-by-side Visual Detections
# ============================================================

# Reload YOLOv8 best model nếu đã bị xóa
try:
    best_yolo
except NameError:
    from ultralytics import YOLO
    best_yolo_path = os.path.join(YOLO_PROJECT, YOLO_RUN_NAME, 'weights', 'best.pt')
    best_yolo = YOLO(best_yolo_path)

compare_ids = val_ids[:4]

fig, axes = plt.subplots(len(compare_ids), 3, figsize=(20, 5 * len(compare_ids)))
fig.suptitle('Ground Truth vs YOLOv8 vs DETR', fontsize=16, fontweight='bold')

for row, img_id in enumerate(compare_ids):
    img_path = os.path.join(IMAGE_DIR, f'{img_id}.png')
    image = Image.open(img_path).convert('RGB')

    # ── Column 0: Ground Truth ──
    ax = axes[row, 0]
    ax.imshow(image)
    for ann in all_annotations[img_id]:
        x1, y1, x2, y2 = ann['bbox']
        color = CLASS_COLORS[ann['class']]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, ann['class'], color=color, fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.7))
    ax.set_title('Ground Truth', fontsize=11)
    ax.axis('off')

    # ── Column 1: YOLOv8 ──
    ax = axes[row, 1]
    ax.imshow(image)
    results = best_yolo(img_path, verbose=False)
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls_id = int(box.cls[0].cpu())
        conf = float(box.conf[0].cpu())
        cls_name = SELECTED_CLASSES[cls_id] if cls_id < len(SELECTED_CLASSES) else f'cls_{cls_id}'
        color = CLASS_COLORS.get(cls_name, 'yellow')
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, f'{cls_name} {conf:.2f}', color=color, fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.7))
    ax.set_title('YOLOv8s', fontsize=11)
    ax.axis('off')

    # ── Column 2: DETR ──
    ax = axes[row, 2]
    ax.imshow(image)
    inputs = detr_processor(images=image, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = detr_model(**inputs)
    target_sizes = torch.tensor([image.size[::-1]], device=device)
    detr_results = detr_processor.post_process_object_detection(
        outputs, threshold=0.5, target_sizes=target_sizes
    )[0]
    for score, label, box in zip(detr_results['scores'], detr_results['labels'], detr_results['boxes']):
        x1, y1, x2, y2 = box.cpu().numpy()
        cls_id = label.item()
        conf = score.item()
        cls_name = SELECTED_CLASSES[cls_id] if cls_id < len(SELECTED_CLASSES) else f'cls_{cls_id}'
        color = CLASS_COLORS.get(cls_name, 'yellow')
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, f'{cls_name} {conf:.2f}', color=color, fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.7))
    ax.set_title('DETR', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.4. FPS Bar Chart & mAP vs FPS Scatter
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FPS Bar Chart
models = ['YOLOv8s', 'DETR']
fps_values = [yolo_fps, detr_fps]
colors_bar = ['#3498db', '#e74c3c']

axes[0].bar(models, fps_values, color=colors_bar, width=0.5, edgecolor='white')
axes[0].set_title('Inference Speed (FPS)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frames per Second')
for i, v in enumerate(fps_values):
    axes[0].text(i, v + 1, f'{v:.1f}', ha='center', fontweight='bold', fontsize=12)
axes[0].grid(alpha=0.3, axis='y')

# mAP vs FPS Scatter
map_values = [yolo_map_results['map'].item(), detr_map_results['map'].item()]

axes[1].scatter(fps_values[0], map_values[0], s=200, c='#3498db', marker='o',
               label='YOLOv8s', zorder=5, edgecolors='black')
axes[1].scatter(fps_values[1], map_values[1], s=200, c='#e74c3c', marker='^',
               label='DETR', zorder=5, edgecolors='black')

axes[1].set_title('mAP@0.50:0.95 vs FPS', fontsize=14, fontweight='bold')
axes[1].set_xlabel('FPS')
axes[1].set_ylabel('mAP@0.50:0.95')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

# Annotate
for i, model_name in enumerate(models):
    axes[1].annotate(model_name, (fps_values[i], map_values[i]),
                     textcoords='offset points', xytext=(10, 10),
                     fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.5. Export Results to CSV
# ============================================================
comparison_df.to_csv('/kaggle/working/detection_comparison_results.csv', index=False)
print('Results saved to /kaggle/working/detection_comparison_results.csv')

# Per-class results
per_class_data = {
    'Class': SELECTED_CLASSES,
    'YOLOv8_AP': yolo_ap,
    'DETR_AP': detr_ap,
}
per_class_df = pd.DataFrame(per_class_data)
per_class_df.to_csv('/kaggle/working/detection_per_class_results.csv', index=False)
print('Per-class results saved to /kaggle/working/detection_per_class_results.csv')
print()
display(per_class_df)

# 7. Discussion & Conclusion

## 7.1. So sanh YOLOv8 vs DETR

### Architecture
| | YOLOv8s (CNN-based) | DETR (Transformer-based) |
|---|---|---|
| **Backbone** | CSPDarknet (pure CNN) | ResNet-50 (CNN) + Transformer Encoder-Decoder |
| **Detection Head** | Anchor-free, multi-scale FPN | Set prediction with bipartite matching (Hungarian algorithm) |
| **Post-processing** | NMS (Non-Maximum Suppression) | Direct set prediction (no NMS needed) |
| **Training epochs** | 50 (fast convergence) | 20 (slow per epoch, limited by Kaggle GPU time) |
| **Pretrained on** | COCO (80 classes) | COCO (91 classes) |

### Training Configuration
- **Cùng điều kiện**: Same dataset (KITTI), same train/val split (80/20, SEED=42), same 7 classes, same GPU (T4)
- **Khác epochs**: YOLOv8 train 50 epochs (~15 min total) vs DETR train 20 epochs (~60 min total). Mặc dù số epochs khác nhau, tổng thời gian training tương đương nhau — đây là thực tế của sự khác biệt giữa CNN và Transformer
- **Cùng evaluation**: torchmetrics MeanAveragePrecision (COCO protocol) cho cả hai mô hình

### CNN vs Transformer Trade-offs
1. **Speed**: YOLOv8 (CNN) nhanh hơn nhiều do local convolution operations. DETR chậm hơn do self-attention O(n^2) trên toàn bộ feature map
2. **Training efficiency**: YOLOv8 hội tụ nhanh (~50 epochs). DETR cần nhiều epochs hơn (gốc paper: 300 epochs trên COCO) do Hungarian matching loss phức tạp
3. **Object size**: DETR có thể tốt hơn với large objects nhờ global attention. YOLOv8 tốt hơn với small objects nhờ multi-scale FPN
4. **Memory**: DETR cần nhiều GPU memory hơn (batch_size=4 vs 16) do attention mechanism
5. **Post-processing**: DETR không cần NMS — set prediction trực tiếp. YOLOv8 vẫn cần NMS

### Limitations
- KITTI dataset có class imbalance lớn (Car chiếm đa số)
- DETR chưa được train đủ epochs (20 vs recommended 300) nên chưa đạt hiệu suất tối ưu
- Chưa so sánh với các variant như YOLOv8m/l hoặc Deformable DETR

### Future Work
- Thử nghiệm **RT-DETR** (Real-Time DETR) — kết hợp tốc độ YOLO với kiến trúc Transformer
- So sánh với **Deformable DETR** — cải thiện tốc độ hội tụ của DETR
- Fine-tune DETR lâu hơn (100+ epochs) để đạt kết quả tốt nhất

In [ ]:
# ============================================================
# Final Summary
# ============================================================
print('=' * 70)
print('              OBJECT DETECTION — FINAL SUMMARY')
print('=' * 70)
print(f'\n  Dataset: KITTI Object Detection')
print(f'  Classes: {NUM_CLASSES} ({SELECTED_CLASSES})')
print(f'  Train: {len(train_ids)} images | Val: {len(val_ids)} images')
print(f'\n  {"Metric":<20} {"YOLOv8s":>12} {"DETR":>12}')
print(f'  {"-"*20} {"-"*12} {"-"*12}')
print(f'  {"mAP@0.50":<20} {yolo_map_results["map_50"].item():>12.4f} {detr_map_results["map_50"].item():>12.4f}')
print(f'  {"mAP@0.50:0.95":<20} {yolo_map_results["map"].item():>12.4f} {detr_map_results["map"].item():>12.4f}')
print(f'  {"FPS":<20} {yolo_fps:>12.1f} {detr_fps:>12.1f}')
print(f'  {"Latency (ms)":<20} {yolo_latency:>12.1f} {detr_latency:>12.1f}')
print(f'\n  Seed: {SEED} | Device: {device}')
print('=' * 70)